# 106 — Compresión de contexto y cachés semánticos

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Cada llamada paga `O(tokens de entrada)` en coste y latencia (*prefill*). Cuatro
palancas:

- **Presupuesto de contexto**: repartir la ventana por valor, no por capacidad:
  instrucciones + herramientas + memoria + top-k + turno.
- **Compresión dura** (LLMLingua, arXiv:2310.05736): descartar tokens de baja
  información según perplejidad de un LM pequeño; 2-10× con poca degradación.
  LongLLMLingua comprime guiado por la pregunta.
- **Compresión blanda**: resumir con LLM (compactación de la clase 105); fluida pero
  puede parafrasear mal un dato crítico.
- **Prompt caching** (exacto): el proveedor cachea el KV del **prefijo idéntico**;
  lectura ≈ 10 % del coste. Consecuencia: prompt ordenado de estable a volátil; un
  token cambiado invalida todo lo que sigue.
- **Caché semántico** (aproximado): reutilizar la respuesta completa si
  `sim(E(q), E(q')) ≥ τ`. τ bajo → falsos aciertos (responder otra pregunta); τ alto →
  caché inútil. Exige invalidación cuando el corpus cambia.

## 🧮 Ejemplo de referencia

Caché semántico con `τ = 0.92`; en caché "¿cómo reinicio mi contraseña?":

```text
q1 "¿cómo restablezco mi contraseña?"  sim 0.96 → HIT correcto
q2 "¿cómo cambio mi contraseña?"       sim 0.93 → HIT frontera (¿mismo procedimiento?)
q3 "¿cómo reinicio mi router?"         sim 0.85 → MISS → pipeline

Economía: 10 000 consultas/día, hit-rate 40 %, $0.004/llamada
  ahorro ≈ 10 000 · 0.40 · 0.004 = $16/día (~$480/mes); latencia 900 → ~50 ms en hit.
Riesgo: 2 % de hits falsos = 80 respuestas equivocadas/día.
```

La decisión real no es técnica sino de dominio: ¿vale el ahorro ese riesgo?

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("retrieval", seed=106)
show(result)


## Reflexión

1. Prompt caching y caché semántico ahorran cosas distintas. ¿Cuál elimina la llamada al LLM y cuál solo abarata el prefill, y por qué pueden (y suelen) convivir?
2. Si subes τ de 0.92 a 0.97, ¿qué pasa con el hit-rate, con los falsos aciertos y con el ahorro neto? ¿Qué datos necesitas para elegir τ de forma defendible?
3. ¿Por qué colocar la fecha actual en la primera línea del system prompt es un error caro con prompt caching, y dónde la colocarías?